# Laboratorio Clase 7: cómo ve una máquina

**Módulo II: Navegando por los componentes de la IA. Sesión 7, 12 de noviembre de 2026**

En la clase vimos que para una máquina una imagen es una matriz de números. En este notebook recorres el camino completo: de los píxeles a una decisión de negocio.

| Misión | Qué vas a hacer | Concepto de la clase |
|---|---|---|
| 1 | Abrir una foto y ver sus números | Píxeles y canales RGB |
| 2 | Aplicar un filtro de bordes a mano | Convolución, la base de las CNN |
| 3 | Detectar objetos con YOLO26 | Detección: etiqueta, caja y confianza |
| 4 | Leer una factura con Claude y validar el resultado | VLM, OCR y salida estructurada |
| 5 | Convertir una detección en una alerta | El problema del "último kilómetro" |

CNN significa red neuronal convolucional. VLM significa modelo de visión y lenguaje. OCR significa reconocimiento óptico de caracteres.

### Cómo usar este notebook

Ejecuta cada celda con **Shift + Enter**. Lee el resultado antes de seguir. Si algo falla, usa **Kernel, Restart y Run All**. No necesitas entender cada línea: lee los comentarios que empiezan con `#`.

Cada misión termina con una pregunta **Para la puesta en común**. Anota tu respuesta: la conversamos al final.

## Paso 0: revisar el ambiente

Este laboratorio tiene su propio ambiente virtual en la carpeta `clase_07-vision/.venv`. Se crea con `uv sync`. La misión 3 descarga una vez el modelo YOLO26n (unos 5 MB). La misión 4 usa la API de Claude si encuentra la variable `ANTHROPIC_API_KEY` en un archivo `.env`; si no la encuentra, usa un modo simulado.

In [ ]:
# Paso 0: importar las herramientas
import importlib.util, subprocess, sys
if "google.colab" in sys.modules and importlib.util.find_spec("ultralytics") is None:
    subprocess.run(["uv", "pip", "install", "--system", "-q", "ultralytics"], check=True)   # Colab no trae YOLO

import warnings
warnings.filterwarnings("ignore")

import base64, io, json, os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from PIL import Image, ImageDraw, ImageFont
import ultralytics
from ultralytics import YOLO

ASSETS = Path(ultralytics.__file__).parent / "assets"  # fotos de ejemplo que trae Ultralytics
SALIDAS = Path("salidas")
SALIDAS.mkdir(exist_ok=True)
print("Fotos disponibles:", sorted(p.name for p in ASSETS.glob("*.jpg")))
print("Ambiente listo.")

## Misión 1: una foto es una tabla de números

Cada píxel guarda tres números entre 0 y 255: rojo (R), verde (G) y azul (B). Vamos a abrir una foto y mirar sus números.

In [ ]:
# Misión 1a: abrir la foto y medir su tamaño
foto = Image.open(ASSETS / "bus.jpg").convert("RGB")
matriz = np.array(foto)

alto, ancho, canales = matriz.shape
print(f"Alto: {alto} píxeles, ancho: {ancho} píxeles, canales: {canales}")
print(f"Total de números: {matriz.size:,}".replace(",", "."))
plt.figure(figsize=(4, 5)); plt.imshow(matriz); plt.axis("off"); plt.show()

In [ ]:
# Misión 1b: un parche de 5 x 5 píxeles del canal rojo
fila, columna = 300, 400
print("Canal R en el parche (fila 300, columna 400):")
print(matriz[fila:fila + 5, columna:columna + 5, 0])

In [ ]:
# Misión 1c: separar los tres canales
fig, ejes = plt.subplots(1, 3, figsize=(10, 4))
for i, (nombre, mapa) in enumerate([("R (rojo)", "Reds"), ("G (verde)", "Greens"), ("B (azul)", "Blues")]):
    ejes[i].imshow(matriz[:, :, i], cmap=mapa)
    ejes[i].set_title(nombre); ejes[i].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Misión 1d: cambiar el brillo es sumar un número a cada píxel
def brillo(cambio=0):
    nueva = np.clip(matriz.astype(int) + cambio, 0, 255).astype(np.uint8)
    plt.figure(figsize=(4, 5)); plt.imshow(nueva); plt.axis("off")
    plt.title(f"Cada número {cambio:+d}"); plt.show()

widgets.interact(brillo, cambio=widgets.IntSlider(value=0, min=-150, max=150, step=10));

Para la máquina, "aclarar una foto" es una suma. Todo lo que hace la visión por computador son operaciones sobre estas tablas.

**Para la puesta en común:** ¿cuántos números por segundo genera una cámara 4K a 30 cuadros por segundo? Calcula 3840 x 2160 x 3 x 30. ¿Qué implica para el costo de almacenar y procesar video?

---

## Misión 2: la convolución, el "ojo" de una CNN

Una CNN no mira la foto completa de una vez. Pasa un **filtro** pequeño (por ejemplo, de 3 x 3) por toda la imagen. En cada posición multiplica y suma. El filtro de Sobel resalta los bordes verticales.

In [ ]:
# Misión 2a: convolución escrita a mano con numpy
gris = np.array(foto.convert("L"), dtype=float)

def convolucion(imagen, filtro):
    ventanas = np.lib.stride_tricks.sliding_window_view(imagen, filtro.shape)
    return (ventanas * filtro).sum(axis=(-2, -1))

sobel_vertical = np.array([[-1, 0, 1],
                           [-2, 0, 2],
                           [-1, 0, 1]])
bordes = np.abs(convolucion(gris, sobel_vertical))

fig, ejes = plt.subplots(1, 2, figsize=(8, 5))
ejes[0].imshow(gris, cmap="gray"); ejes[0].set_title("Foto en gris")
ejes[1].imshow(bordes, cmap="gray"); ejes[1].set_title("Bordes verticales (Sobel)")
for e in ejes: e.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Misión 2b: elige un filtro y mira el efecto
filtros = {
    "Bordes verticales": sobel_vertical,
    "Bordes horizontales": sobel_vertical.T,
    "Desenfoque": np.ones((3, 3)) / 9,
    "Enfoque": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]),
}

def probar(nombre="Bordes verticales"):
    filtro = filtros[nombre]
    salida = convolucion(gris, filtro)
    if "Bordes" in nombre:
        salida = np.abs(salida)
    print("Filtro:"); print(np.round(filtro, 2))
    plt.figure(figsize=(4, 5)); plt.imshow(salida, cmap="gray"); plt.axis("off"); plt.title(nombre); plt.show()

widgets.interact(probar, nombre=list(filtros));

Aquí tú elegiste los números del filtro. Una CNN **aprende** esos números a partir de miles de ejemplos etiquetados. Las primeras capas aprenden bordes, las siguientes texturas y las últimas objetos completos. Los modelos más recientes, como los ViT (Vision Transformer), cortan la imagen en parches y usan atención, la misma idea de la clase 6.

**Para la puesta en común:** si una CNN aprende sus filtros con ejemplos, ¿qué pasa si en los ejemplos nunca aparece un casco de color naranjo?

---

## Misión 3: detección de objetos con YOLO26

YOLO26 es la versión de enero de 2026 de la familia YOLO (You Only Look Once) de Ultralytics. Está pensada para correr en equipos pequeños, como una cámara o un computador industrial. Viene entrenada con 80 clases comunes: persona, auto, bus, perro, entre otras.

In [ ]:
# Misión 3a: cargar el modelo (la primera vez se descarga)
ruta = SALIDAS / "yolo26n.pt"
modelo = YOLO(ruta if ruta.exists() else "yolo26n.pt")
if not ruta.exists() and Path("yolo26n.pt").exists():
    Path("yolo26n.pt").replace(ruta)   # guarda los pesos en salidas/ para la próxima vez
print(f"Clases que conoce: {len(modelo.names)}. Ejemplos: {list(modelo.names.values())[:8]}")

In [ ]:
# Misión 3b: detectar objetos y mostrar etiqueta, confianza y caja
def detectar(nombre_foto, confianza=0.25, mostrar=True):
    r = modelo(ASSETS / nombre_foto, conf=confianza, verbose=False)[0]
    filas = [
        {"etiqueta": r.names[int(c)], "confianza": round(float(p), 2), "caja": [round(float(v)) for v in b]}
        for c, p, b in zip(r.boxes.cls, r.boxes.conf, r.boxes.xyxy)
    ]
    if mostrar:
        plt.figure(figsize=(6, 6)); plt.imshow(r.plot()[:, :, ::-1]); plt.axis("off"); plt.show()
    return filas

for d in detectar("bus.jpg"):
    print(f"{d['etiqueta']:10} confianza {d['confianza']:.2f}  caja {d['caja']}")

In [ ]:
# Misión 3c: el umbral de confianza es una decisión de negocio
def umbral(confianza=0.25):
    filas = detectar("zidane.jpg", confianza)
    print(f"Con umbral {confianza:.2f} el modelo reporta {len(filas)} objetos:")
    for d in filas:
        print(f"  {d['etiqueta']:10} {d['confianza']:.2f}")

widgets.interact(umbral, confianza=widgets.FloatSlider(value=0.25, min=0.05, max=0.95, step=0.05));

Con un umbral bajo, el modelo reporta más objetos, pero algunos son falsos. Con un umbral alto, reporta menos, pero puede omitir objetos reales. No hay un umbral correcto: depende del costo de cada error.

**Para la puesta en común:** en una faena minera, ¿qué es más caro, una falsa alarma de "persona sin casco" o una persona sin casco que nadie detecta? ¿Qué umbral elegirías?

---

## Misión 4: leer una factura con un VLM

Un OCR clásico entrega letras sueltas. Un VLM entiende qué significa cada campo. Vamos a dibujar una factura ficticia, enviarla a Claude Haiku 5.5 y pedir un objeto con campos fijos. Después **validamos con código**: el modelo puede equivocarse.

Primero, la regla del RUT (Rol Único Tributario). El dígito verificador se calcula con el algoritmo módulo 11.

In [ ]:
# Misión 4a: calcular y validar el dígito verificador del RUT
def digito_verificador(cuerpo: int) -> str:
    suma, factor = 0, 2
    for d in reversed(str(cuerpo)):
        suma += int(d) * factor
        factor = 2 if factor == 7 else factor + 1
    resto = 11 - suma % 11
    return {11: "0", 10: "K"}.get(resto, str(resto))

def rut_valido(rut: str) -> bool:
    limpio = rut.replace(".", "").replace("-", "").upper()
    return limpio[:-1].isdigit() and digito_verificador(int(limpio[:-1])) == limpio[-1]

assert digito_verificador(11111111) == "1" and digito_verificador(76086428) == "5"
print("76.086.428-5 es válido:", rut_valido("76.086.428-5"))
print("76.086.428-3 es válido:", rut_valido("76.086.428-3"))

In [ ]:
# Misión 4b: dibujar una factura ficticia
def iva(neto: int) -> int:
    return (neto * 19 + 50) // 100   # 19% redondeado al peso, con enteros: sin errores de decimales

def miles(n):
    return f"{n:,}".replace(",", ".")

cuerpo = 76543210
RUT_EMISOR = f"{miles(cuerpo)}-{digito_verificador(cuerpo)}"
NETO = 1_250_000
IVA = iva(NETO)
TOTAL = NETO + IVA

lineas = [
    ("Comercial Los Aromos SpA", 34),
    (f"RUT: {RUT_EMISOR}", 24),
    ("FACTURA ELECTRÓNICA N° 004517", 28),
    ("Fecha de emisión: 03-11-2026", 22),
    ("Cliente: Minera Cordillera Ltda.", 22),
    ("Detalle: 50 cascos de seguridad clase E", 22),
    (f"Monto neto: $ {miles(NETO)}", 24),
    (f"IVA 19%: $ {miles(IVA)}", 24),
    (f"Total: $ {miles(TOTAL)}", 30),
]
factura_img = Image.new("RGB", (760, 520), "white")
lapiz = ImageDraw.Draw(factura_img)
y = 30
for texto, tam in lineas:
    lapiz.text((40, y), texto, fill="black", font=ImageFont.load_default(size=tam))
    y += tam + 22
lapiz.rectangle([20, 15, 740, 505], outline="gray", width=2)
factura_img.save(SALIDAS / "factura_ficticia.png")
plt.figure(figsize=(7, 5)); plt.imshow(factura_img); plt.axis("off"); plt.show()

In [ ]:
# Misión 4c: el formato que exigimos al modelo
from pydantic import BaseModel

class Factura(BaseModel):
    emisor: str
    rut_emisor: str
    numero: int
    fecha: str
    neto: int
    iva: int
    total: int

USAR_CLAUDE = True  # cambia a False para usar el modo simulado

def cargar_env():
    try:
        from dotenv import load_dotenv
    except ImportError:   # por ejemplo, en Colab
        return False
    aqui = Path.cwd().resolve()
    for carpeta in [aqui, *list(aqui.parents)[:5]]:
        env = carpeta / ".env"
        if env.is_file():
            load_dotenv(env); return True
    return False

if USAR_CLAUDE:
    cargar_env()
hay_clave = USAR_CLAUDE and bool(os.environ.get("ANTHROPIC_API_KEY"))
print("Modo:", "Claude Haiku 5.5 por API" if hay_clave else "simulado (sin clave de API)")

In [ ]:
# Misión 4d: enviar la imagen y recibir un objeto Factura
def factura_simulada():
    # ponytail: respuesta fija para clases sin clave; trae un error a propósito para la misión 4e
    return Factura(emisor="Comercial Los Aromos SpA", rut_emisor=RUT_EMISOR, numero=4517,
                   fecha="03-11-2026", neto=NETO, iva=IVA, total=TOTAL - 1000)

def leer_factura(imagen):
    if not hay_clave:
        return factura_simulada()
    import anthropic
    buffer = io.BytesIO(); imagen.save(buffer, format="PNG")
    datos = base64.standard_b64encode(buffer.getvalue()).decode()
    try:
        respuesta = anthropic.Anthropic().messages.parse(
            model="claude-haiku-5-5",
            max_tokens=1024,
            messages=[{"role": "user", "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": datos}},
                {"type": "text", "text": "Extrae los datos de esta factura chilena. Montos en pesos, como enteros sin puntos."},
            ]}],
            output_format=Factura,
        )
    except anthropic.APIError as error:   # sin internet, clave inválida o servicio caído
        print(f"La API falló ({type(error).__name__}). Sigo en modo simulado.")
        return factura_simulada()
    if respuesta.stop_reason == "refusal" or respuesta.parsed_output is None:
        raise RuntimeError(f"El modelo no entregó una factura (stop_reason={respuesta.stop_reason}).")
    print(f"Tokens de entrada: {respuesta.usage.input_tokens}, de salida: {respuesta.usage.output_tokens}")
    return respuesta.parsed_output

factura = leer_factura(factura_img)
print(json.dumps(factura.model_dump(), indent=2, ensure_ascii=False))

In [ ]:
# Misión 4e: el humano en el ciclo. Validar con reglas, no con confianza
def validar(f: Factura):
    reglas = {
        "RUT con dígito verificador correcto": rut_valido(f.rut_emisor),
        "IVA = 19% del neto": f.iva == iva(f.neto),
        "Neto + IVA = total": f.neto + f.iva == f.total,
    }
    for regla, ok in reglas.items():
        print(f"{'OK   ' if ok else 'FALLA'}  {regla}")
    return all(reglas.values())

if validar(factura):
    print("Factura lista para el ERP (sistema de planificación de recursos empresariales).")
else:
    print("Factura a revisión humana antes de pagar.")

El modelo entrega un objeto con campos fijos, listo para un sistema. Pero la salida estructurada asegura el **formato**, no la **verdad**. Las reglas de negocio se verifican con código. En el modo simulado, el total trae un error de $1.000 a propósito: la validación lo atrapa.

**Para la puesta en común:** ¿qué otras reglas agregarías antes de pagar una factura leída por IA? Piensa en el SII (Servicio de Impuestos Internos), en proveedores nuevos y en montos fuera de lo habitual.

---

## Misión 5: el "último kilómetro"

Un detector entrega un JSON. El negocio necesita una acción. Vamos a escribir la "gasfitería": una regla que convierte el conteo de personas en una alerta.

In [ ]:
# Misión 5: de la detección a la acción
AFORO_MAXIMO = 3  # personas permitidas en la zona de carga

def revisar_zona(nombre_foto, aforo=AFORO_MAXIMO):
    detecciones = detectar(nombre_foto, confianza=0.5, mostrar=False)
    personas = sum(d["etiqueta"] == "person" for d in detecciones)
    evento = {"camara": nombre_foto, "personas": personas, "aforo": aforo, "alerta": personas > aforo}
    print(json.dumps(evento, ensure_ascii=False))
    if evento["alerta"]:
        print(f"ALERTA al supervisor: {personas} personas en la zona de carga (máximo {aforo}).")
    else:
        print("Zona dentro del aforo. Sin acción.")

def probar_aforo(aforo=AFORO_MAXIMO):
    for nombre in ["bus.jpg", "zidane.jpg"]:
        revisar_zona(nombre, aforo)

widgets.interact(probar_aforo, aforo=widgets.IntSlider(value=AFORO_MAXIMO, min=1, max=6));

El modelo solo cuenta personas. No sabe quiénes son. **Detectar** una persona no es lo mismo que **identificarla**. Esa diferencia importa: la Ley 21.719 de protección de datos personales trata los datos biométricos, como el rostro, como datos sensibles. La ley rige desde el 1 de diciembre de 2026; un proyecto de prórroga al 15 de octubre de 2027 está en trámite en el Congreso.

**Para la puesta en común:** su empresa quiere pasar de "contar personas" a "reconocer quién entra". ¿Qué cambia en el riesgo legal, en el consentimiento y en la conversación con los trabajadores?

---

## Cierre

Hoy recorriste el camino completo:

1. **Píxeles:** una foto es una tabla de números.
2. **Convolución:** filtros que encuentran patrones; una CNN los aprende.
3. **Detección:** etiqueta, caja y confianza; el umbral es una decisión de negocio.
4. **VLM:** de la imagen al dato estructurado, siempre con validación.
5. **Último kilómetro:** el valor está en la acción, no en el modelo.

**Sigue con la parte 2:** `clase_07-lab_redes.ipynb` abre una red neuronal por dentro.